In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu tqdm pandas

import os
import json
import random
import re
import torch
from tqdm import tqdm
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 下载 HaluEval 数据
import os

if not os.path.exists("HaluEval"):
    print("正在克隆 HaluEval 仓库...")
    !git clone https://github.com/RUCAIBox/HaluEval.git --depth 1
else:
    print("HaluEval 仓库已存在，跳过克隆")

data_path = "HaluEval/data/qa_data.json"

# 验证文件是否存在
if os.path.exists(data_path):
    print(f"✅ 数据集已就绪: {data_path}")
else:
    print(f"❌ 文件仍不存在: {data_path}")
    # 如果仓库结构变了，尝试找实际文件
    !find HaluEval -name "*qa_data*" -type f
import json

data_path = "HaluEval/data/qa_data.json"

full_data = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # 跳过空行
            try:
                item = json.loads(line)
                full_data.append(item)
            except json.JSONDecodeError:
                continue  # 跳过可能的损坏行

print(f" Data loading complete, {len(full_data)} samples in total")
print("Sample Example (First 1）：")
print(json.dumps(full_data[0], ensure_ascii=False, indent=2) if full_data else "No data")

# 参数设置
NUM_SAMPLES = 50
RANDOM_SEED = 42
TOP_K = 2                 # RAG 检索返回几条知识
MAX_NEW_TOKENS = 256

random.seed(RANDOM_SEED)

# 随机抽取实验样本
samples = random.sample(full_data, NUM_SAMPLES)
print(f"本次实验使用 {NUM_SAMPLES} 条样本进行消融测试")

# 构建 RAG 检索器（全局知识库）
corpus = [item["knowledge"] for item in full_data]  # 用全部 knowledge 做知识库

embedder = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = embedder.encode(corpus, convert_to_tensor=False, show_progress_bar=True)
corpus_embeddings = corpus_embeddings.astype(np.float32)

# FAISS 索引
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_embeddings)

def retrieve(question: str, k: int = TOP_K) -> str:
    """检索最相关的 k 条知识"""
    q_emb = embedder.encode([question]).astype(np.float32)
    _, indices = index.search(q_emb, k)
    retrieved_docs = [corpus[i] for i in indices[0]]
    # 为 Citation 准备带编号的上下文
    numbered = [f"[Source {i+1}] {doc}" for i, doc in enumerate(retrieved_docs)]
    return "\n\n".join(numbered)

print(" RAG index construction complete!")

# 加载 Qwen2.5-7B-Instruct（4bit 量化）
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

print("Model loading complete!")

# 提示词模板（6 组对照）
def build_prompt(question: str, context: str | None, prompt_type: str) -> str:
    if prompt_type == "standard":
        if context:
            return f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {question}
Answer:"""
        else:
            return f"""Answer the following question concisely.

Question: {question}
Answer:"""

    elif prompt_type == "citation":
        if context:
            return f"""Based on the provided context, answer the question and cite sources using [Source 1], [Source 2], etc.

Context:
{context}

Question: {question}
Answer (with citations):"""
        else:
            return f"""Answer the following question concisely based on your knowledge.

Question: {question}
Answer:"""

    elif prompt_type == "self_reflection":
        base = f"""Question: {question}
"""
        if context:
            base += f"""Context:
{context}
"""
        base += """Think step by step:
1. Understand the question and any provided context.
2. Generate the answer.
3. Reflect: Is this answer fully supported by the context or your knowledge? If not, revise it.
Final Answer and Reflection:"""
        return base

# 生成答案函数
def generate_answer(messages: list) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=0.3,      # 较低温度，更稳定
        top_p=0.85,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# 6组对照实验配置
configs = {
    "Standard": {"rag": False, "prompt_type": "standard"},
    "Citation": {"rag": False, "prompt_type": "citation"},
    "Self-Reflection": {"rag": False, "prompt_type": "self_reflection"},
    "Standard + RAG": {"rag": True, "prompt_type": "standard"},
    "Citation + RAG": {"rag": True, "prompt_type": "citation"},
    "Self-Reflection + RAG": {"rag": True, "prompt_type": "self_reflection"},
}

# 运行实验 + 评估
print("\n开始 6 组消融实验")

results = []
system_prompt = "You are a helpful, truthful and accurate assistant."

for config_name, cfg in tqdm(configs.items(), desc="实验进度"):
    is_rag = cfg["rag"]
    ptype = cfg["prompt_type"]

    for sample in tqdm(samples, desc=config_name, leave=False):
        question = sample["question"]
        right_answer = sample["right_answer"]

        # RAG 检索
        context = retrieve(question, k=TOP_K) if is_rag else None

        # 构建提示
        user_content = build_prompt(question, context, ptype)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]

        # 生成答案
        generated = generate_answer(messages)

        # 简单准确率评估（right_answer 是否出现在生成答案中）
        gen_norm = re.sub(r'[^a-z0-9\s]', '', generated.lower()).strip()
        right_norm = re.sub(r'[^a-z0-9\s]', '', right_answer.lower()).strip()
        is_correct = (right_norm in gen_norm) or (len(right_norm) > 2 and right_norm.split()[0] in gen_norm)

        results.append({
            "config": config_name,
            "question": question,
            "right_answer": right_answer,
            "generated_answer": generated,
            "correct": is_correct
        })

# 输出结果
df = pd.DataFrame(results)
summary = df.groupby("config")["correct"].agg(["mean", "count"]).reset_index()
summary = summary.rename(columns={"mean": "accuracy"})
summary = summary.sort_values("accuracy", ascending=False)

print("\n" + "="*60)
print("消融实验结果")
print("="*60)
print(summary.to_string(index=False))

best_config = summary.iloc[0]["config"]
best_acc = summary.iloc[0]["accuracy"]
print(f"\n最优组合是：{best_config} （准确率 {best_acc:.1%}）")

# 保存详细结果
df.to_csv("halueval_ablation_results.csv", index=False)
print("详细结果已保存为 halueval_ablation_results.csv，可下载查看每条生成答案")

HaluEval 仓库已存在，跳过克隆
✅ 数据集已就绪: HaluEval/data/qa_data.json
 Data loading complete, 10000 samples in total
Sample Example (First 1）：
{
  "knowledge": "Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.",
  "question": "Which magazine was started first Arthur's Magazine or First for Women?",
  "right_answer": "Arthur's Magazine",
  "hallucinated_answer": "First for Women was started first."
}
本次实验使用 50 条样本进行消融测试


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loading complete!

开始 6 组消融实验


实验进度: 100%|██████████| 6/6 [49:17<00:00, 492.92s/it]


消融实验结果
               config  accuracy  count
Self-Reflection + RAG      0.80     50
       Citation + RAG      0.76     50
       Standard + RAG      0.74     50
             Citation      0.46     50
      Self-Reflection      0.40     50
             Standard      0.40     50

最优组合是：Self-Reflection + RAG （准确率 80.0%）
详细结果已保存为 halueval_ablation_results.csv，可下载查看每条生成答案
